# 01 — Exploring the metro housing panel

Goal: understand the data before modeling. How many metros, how far back, what do price paths look like, and how much does 12-month growth vary across metros and over time?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent  # notebook lives in notebooks/, project root is one up
sys.path.insert(0, str(ROOT))
from src.features import add_baseline_features

panel = pd.read_parquet(ROOT / "data" / "processed" / "panel.parquet")
panel = add_baseline_features(panel)

In [ ]:
print(f"{len(panel):,} rows, {panel['region_id'].nunique()} metros")
print(f"Dates: {panel['date'].min().date()} to {panel['date'].max().date()}")
panel.head()

## Coverage: how many metros have data each year?

In [ ]:
coverage = panel.groupby(panel["date"].dt.year)["region_id"].nunique()
coverage.plot(kind="bar", figsize=(12, 3), title="Metros with data, by year")
plt.ylabel("Metros")
plt.show()

## Home values for the five largest metros

In [ ]:
largest = panel.groupby("region_id")["size_rank"].first().nsmallest(5).index
fig, ax = plt.subplots(figsize=(12, 5))
for _, metro in panel[panel["region_id"].isin(largest)].groupby("metro"):
    ax.plot(metro["date"], metro["zhvi"], label=metro["metro"].iloc[0])
ax.set_title("Zillow Home Value Index")
ax.set_ylabel("Typical home value ($)")
ax.legend()
plt.show()

## 12-month growth over time: median metro and spread across metros

In [ ]:
by_month = panel.groupby("date")["growth_12m"].quantile([0.1, 0.5, 0.9]).unstack() * 100
fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(by_month.index, by_month[0.1], by_month[0.9], alpha=0.3, label="10th-90th percentile")
ax.plot(by_month.index, by_month[0.5], label="Median metro")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("12-month home value growth across metros")
ax.set_ylabel("Percent")
ax.legend()
plt.show()

## Macro context: mortgage rates and unemployment

In [ ]:
macro = panel.groupby("date")[["mortgage_rate_30y", "unemployment_rate"]].first()
macro.plot(subplots=True, figsize=(12, 5), title=["30-year mortgage rate (%)", "Unemployment rate (%)"])
plt.show()

## Notes

Write 3-5 observations here after running the cells, for example: when coverage becomes complete, how the 2008 crash and 2020-22 boom show up, how wide the spread across metros is, and how mortgage rates moved alongside growth. These notes become the "Data" section of your README.